In [ ]:
import requests
import pandas as pd
import numpy as np
import io
from bs4 import BeautifulSoup
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

# ==========================================
# 1. OBSERVATORY LOCATION FUNCTION
# ==========================================
def get_observatory_location(iau_code):
    url_obs = "https://minorplanetcenter.net/iau/lists/ObsCodes.html"
    print(f"--- Downloading official MPC list to search for code {iau_code}... ---")
    r = requests.get(url_obs)
    r.raise_for_status()
    

    for line in r.text.split('\n'):
        if line.startswith(iau_code):
            parts = line.split()
            if len(parts) >= 4:
                long_deg = float(parts[1])
                cos_phi = float(parts[2])
                sin_phi = float(parts[3])
                lat_rad = np.arctan2(sin_phi, cos_phi)
                lat_deg = np.degrees(lat_rad)
                print(f"Observatory found: Lat {lat_deg:.4f}, Lon {long_deg:.4f}")
                return EarthLocation(lat=lat_deg * u.deg, lon=long_deg * u.deg, height=0 * u.m)
    
    print(f"Code {iau_code} not found.")
    print(long_deg , lat_def)
    return None

# ==========================================
# 2. FILTERING FUNCTION
# ==========================================
def filter_visible_objects(df, location):
    print("\n--- Calculating visibility (this may take a while) ---")
    visible_objects = []
    
    # Check next 24h (15 min steps)
    current_time = Time.now()
    delta_time = np.linspace(0, 24, 96) * u.hour 
    times_grid = current_time + delta_time
    
    frame_altaz = AltAz(obstime=times_grid, location=location)

    for index, row in df.iterrows():
        try:
            ra_raw = str(row['R.A.']).strip()
            dec_raw = str(row['Decl.']).strip()
            
            ra_txt = ra_raw.replace(" ", "h", 1) if "h" not in ra_raw else ra_raw
            if "m" not in ra_txt and "h" in ra_txt: ra_txt += "m"
            if "h" not in ra_txt: ra_txt += "h"

            dec_txt = dec_raw.replace(" ", "d", 1) if "d" not in dec_raw else dec_raw
            if "m" not in dec_txt and "d" in dec_txt: dec_txt += "m"
            if "d" not in dec_txt: dec_txt += "d"

            coord = SkyCoord(ra=ra_txt, dec=dec_txt, unit=(u.hourangle, u.deg))
            altaz = coord.transform_to(frame_altaz)
            altitudes = altaz.alt.degree
            
            # Logic: > 2 points (approx 30 mins) above 10 degrees
            visible_points = np.sum(altitudes > 10)
            
            if visible_points >= 2:
                row['Visible_Minutes'] = visible_points * 15
                row['Max_Alt'] = round(np.max(altitudes), 1)
                visible_objects.append(row)
                
        except Exception:
            continue 

    return pd.DataFrame(visible_objects)

# ==========================================
# 3. MAIN EXECUTION
# ==========================================

# Settings
OBSERVATORY_CODE = "Y28"
user_choice = input("Enter 'NEOCP' or 'PCCP' (Default: NEOCP): ").strip().upper()
PAGE_TYPE = "PCCP" if user_choice == "PCCP" else "NEOCP"
url = "https://minorplanetcenter.net/iau/NEO/toconfirm_tabular.html" if PAGE_TYPE == "NEOCP" else "https://minorplanetcenter.net/iau/NEO/pccp_tabular.html"

# Download
print(f"--- Downloading data from: {PAGE_TYPE} ---")
headers = {"User-Agent": "Mozilla/5.0"}
df_raw = None

try:
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.content, 'html.parser')
    table_html = soup.find('table', {'class': 'tablesorter'})
    
    if table_html:
        for hidden in table_html.find_all('span', style=lambda x: x and 'display:none' in x): hidden.decompose()
        for chk in table_html.find_all('input'): chk.decompose()
        df_raw = pd.read_html(io.StringIO(str(table_html)), flavor='bs4')[0]
        df_raw.columns = [c.strip() for c in df_raw.columns]
        df_raw = df_raw.dropna(axis=1, how='all')
    else:
        print("HTML table not found.")

except Exception as e:
    print(f"Download error: {e}")

df_filtered = pd.DataFrame() # Initialize empty

# Filter Logic
if df_raw is not None and not df_raw.empty:
    local_obs = get_observatory_location(OBSERVATORY_CODE)
    
    if local_obs is not None:
        print(f"\nTotal objects downloaded: {len(df_raw)}")
        print(f"Filtering objects with Alt > 10° for ~30 min at {OBSERVATORY_CODE}...")
        
        df_filtered = filter_visible_objects(df_raw, local_obs)
        
        if not df_filtered.empty:
            df_filtered = df_filtered.sort_values(by='Max_Alt', ascending=False)
            
            # Show summary table
            cols = ['Temp Desig', 'R.A.', 'Decl.', 'V', 'Visible_Minutes', 'Max_Alt']
            final_cols = [c for c in cols if c in df_filtered.columns]
            
            print("\n" + "="*60)
            print(f"OBSERVABLE OBJECTS SUMMARY ({OBSERVATORY_CODE})")
            print("="*60)
            pd.set_option('display.max_rows', None)
            print(df_filtered[final_cols].to_string(index=False))
            
            # ==========================================
            # 4. INTERACTIVE DETAILS (NOVA PARTE)
            # ==========================================
            while True:
                print("\n" + "-"*60)
                target = input("Enter the 'Temp Desig' to see full details (or '0' to exit): ").strip()
                
                if target == '0':
                    print("Exiting...")
                    break
                
                # Filter the filtered dataframe to find the specific row
                # We use .values to check exact match or substring
                obj_row = df_filtered[df_filtered['Temp Desig'] == target]
                
                if not obj_row.empty:
                    print(f"\nDETAILS FOR OBJECT: {target}")
                    print("="*30)
                    # iloc[0] selects the first match (should be unique)
                    # print(series) displays index (column name) vs value
                    print(obj_row.iloc[0]) 
                else:
                    print(f"Object '{target}' not found in the visible list. Check spelling (case sensitive).")

        else:
            print("\nNo objects visible with current criteria.")
    else:
        print("Failed to obtain observatory coordinates.")
else:
    print("No data.")

Enter 'NEOCP' or 'PCCP' (Default: NEOCP):  NEOCP


--- Downloading data from: NEOCP ---
--- Downloading official MPC list to search for code Y28... ---
Observatory found: Lat -8.7308, Lon 321.3126

Total objects downloaded: 147
Filtering objects with Alt > 10° for ~30 min at Y28...

--- Calculating visibility (this may take a while) ---

OBSERVABLE OBJECTS SUMMARY (Y28)
Temp Desig    R.A.  Decl.    V  Visible_Minutes  Max_Alt
   ST26A74 10 11.9 -09 20 20.9              645     88.8
   P12kSDl 05 01.0 -09 46 21.3              645     88.5
   P12ks7y 11 10.6 -06 09 21.4              630     87.2
   P22khrl 10 43.2 -05 42 20.7              645     87.0
   6AK1721 09 09.6 -05 29 19.7              630     86.6
   A11xOga 08 05.9 -14 57 18.1              660     83.5
   P12kIwP 11 00.6 -01 58 22.3              645     83.4
   P12kSey 09 40.8 -15 16 22.2              645     83.3
   ST26A31 07 22.9 -00 17 24.7              630     81.5
   ZTF10Am 12 48.9 -17 03 23.9              645     81.5
   ZTF10Ak 18 13.0 +00 14 32.3              645    